In [9]:
import random
from datetime import datetime

import mysql.connector
from faker import Faker
from pydeequ.analyzers import *
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pydeequ.analyzers import AnalysisRunner, Size, Completeness, ApproxCountDistinct
from pydeequ.repository import FileSystemMetricsRepository, ResultKey

In [10]:
Faker.seed(42)
fake = Faker(['ko_KR', 'en_US'])

In [11]:
spark = SparkSession.builder \
    .appName("PyDeequ Example") \
    .master("spark://localhost:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "mmix") \
    .config("spark.hadoop.fs.s3a.secret.key", "mmixmmix") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()
repo_path = "s3a://mmix-prod-dataengineer-datalakehouse/metrics/orders/metrics.json"
repository = FileSystemMetricsRepository(spark, repo_path)
resultKey = ResultKey(spark, ResultKey.current_milli_time(), {"pipeline": "orders_etl", "dataset": "orders", "env": "dev"})

In [12]:
data = [{
    "id": i + 1,
    "name": fake.name(),
    "age": fake.random_int(min=20, max=65),
    "weigh": fake.random_int(min=10, max=250),
    "height": fake.random_int(min=10, max=250),
    "gender": random.choice(['남성', '여성']),
    "address": fake.address(),
    "job": fake.job(),
    "email": fake.email()
} for i in range(100)]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("weight", IntegerType(), True),
    StructField("height", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("address", StringType(), True),
    StructField("job", StringType(), True),
    StructField("email", StringType(), True)
])

In [13]:
df = spark.createDataFrame(data=data, schema=schema)

In [14]:
check = (
    Check(spark, CheckLevel.Error, "Basic data checks")
    .hasSize(lambda x: x == 100)
    .isComplete("id")
    .isComplete("name")
    .isComplete("age")
    .isComplete("gender")
    .isComplete("email")
)

result = (VerificationSuite(spark).onData(df).addCheck(check).run())

In [15]:
analysis_runner = AnalysisRunner(spark)
analysis_result = (
    analysis_runner.onData(df)
    .addAnalyzer(Size())
    .addAnalyzer(Completeness("id"))
    .addAnalyzer(Completeness("name"))
    .addAnalyzer(Completeness("age"))
    .addAnalyzer(Correlation("height", "weight"))
    .addAnalyzer(Completeness("gender"))
    .addAnalyzer(Completeness("address"))
    .addAnalyzer(Completeness("job"))
    .addAnalyzer(Completeness("email"))
    .useRepository(repository)
    .saveOrAppendResult(resultKey)
    .run())

In [16]:
result_df = AnalyzerContext.successMetricsAsDataFrame(spark, analysis_result) \
    .withColumn("run_name", lit("daily_batch")) \
    .withColumn("run_id", lit(f"daily_batch_{datetime.now().strftime('%Y%m%d%H%M%S')}")) \
    .withColumn("logical_datetime", lit(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"))